# 윤정 추출 모델 — KoELECTRA 이진 분류 파인튜닝 (A단계)

**목표**: 가정통신문 문장을 받아 `할 일·중요 일정(1)` vs `노이즈(0)` 로 이진 분류

| 항목 | 내용 |
|------|------|
| 모델 | `monologg/koelectra-small-v3-discriminator` (CPU 시연용 small 변형) |
| 데이터 | `v3_dual_labeled_clean.jsonl` — 문장 단위 `{text, is_todo, is_title}`, 27,800행 |
| 출력 클래스 | 0: 노이즈, 1: 할 일·중요 일정 |
| 카테고리 분류·중요도 | **B단계(경이 모델)** 로 완전 위임 — 이 노트북에서 제거됨 |

**실행 방법**
1. `런타임` → `런타임 유형 변경` → **T4 GPU** 선택
2. 셀을 위에서부터 순서대로 실행 (`Shift + Enter`)
3. 마지막 셀에서 `koelectra-binary.zip` 다운로드 → `model/extraction/checkpoints/koelectra-binary/` 에 압축 풀기

## 1. 라이브러리 설치
코랩 기본 transformers는 버전이 오래됐으므로 새로 설치합니다.

In [ ]:
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.34.0 evaluate==0.4.3 scikit-learn

## 2. GPU 확인
T4 GPU가 잡혔는지 확인합니다. CPU만 보이면 런타임 유형을 바꿔야 합니다.

In [ ]:
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU 이름:', torch.cuda.get_device_name(0))

## 3. 데이터 로드

`v3_dual_labeled_clean.jsonl` 포맷: `{"text": str, "is_todo": bool, "is_title": bool}`

- `is_todo` 필드만 라벨로 사용 (`is_title` 은 무시)
- 10자 미만 / 5,000자 초과는 clean 파일에서 이미 제거됨
- 정제 후 약 27,800행

In [ ]:
import json
import re
from collections import Counter

# ── predict.py 와 동일한 패턴 유지 (변경 시 양쪽 동기화 필요) ────────────────
_HEADER_ONLY = re.compile(
    r'^[^.,!?~]{2,40}(안내|공지|알림|공개수업|상담|학습|행사|일정)\s*$'
)

_OCR_LINE_NOISE = re.compile(
    r'https?://'
    r'|^www\.'
    r'|☎\s*\d'
    r'|^\d{1,2}:\d{2}\s*[~\-–]\s*\d{1,2}:\d{2}'
    r'|^[→←↑↓]+\s*$'
)

NON_TODO_PATTERNS: list[str] = [
    r'^학부모님\s*안녕하십니까',
    r'^안녕하십니까',
    r'^학부모님\s*안녕하세요',
    r'^안녕하세요',
    r'^.*님\s*안녕하(세요|십니까)',
    r'^학부모님께\s*안내드립니다',
    r'^학부모님께\s*드립니다',
    r'안내드립니다\s*\.?\s*$',
    r'드립니다\s*\.?\s*$',
    r'^[^.,!?]{1,30}\s*안내\s*$',
    r'서울갈산초등학교장$',
    r'교장$',
    r'^\d{4}\.\s*\d{1,2}\.\s*\d{1,2}\.?\s*$',
    r'담당\s*[:：]',
    r'^\(.\s*\d{4}-\d{4}',
    r'^08\d{3}\s*서울특별시',
    r'공익제보센터',
    r'자살예방상담',
    r'청소년상담',
]

# ── 데이터 로드 ───────────────────────────────────────────────────────────────
# v3_dual_labeled_clean.jsonl: {text, is_todo, is_title} 문장 단위 포맷
# is_title 은 이 노트북에서 사용하지 않음 (is_todo 분류기만 학습)
with open('v3_dual_labeled_clean.jsonl', encoding='utf-8') as f:
    rows = [json.loads(line) for line in f if line.strip()]

texts: list[str] = []
labels: list[int] = []

for row in rows:
    text = str(row.get('text', '') or '').strip()
    if len(text) < 7:
        continue
    texts.append(text)
    labels.append(int(bool(row.get('is_todo', False))))

cnt = Counter(labels)
print(f'총 문장 수  : {len(texts)}')
print(f'라벨 분포:')
print(f'  0 (노이즈) : {cnt[0]}  ({cnt[0]/len(labels)*100:.1f}%)')
print(f'  1 (할 일)  : {cnt[1]}  ({cnt[1]/len(labels)*100:.1f}%)')


## 4. Train/Val 분할

**stratified split** 으로 이진 라벨 비율을 맞춰서 나눕니다.  
데이터가 적을 경우 노이즈/할 일 비율이 검증셋에서 무너지는 것을 방지합니다.

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

print(f'학습셋: {len(train_texts)}')
print(f'  양성(할 일): {sum(train_labels)}, 음성(노이즈): {len(train_labels) - sum(train_labels)}')
print(f'검증셋: {len(val_texts)}')
print(f'  양성(할 일): {sum(val_labels)}, 음성(노이즈): {len(val_labels) - sum(val_labels)}')

## 5. 토크나이저 + 데이터셋

small 변형은 base 대비 파라미터가 작아 CPU 추론 속도가 약 2배 빠릅니다.  
가정통신문 한국어 문장 평균 30~80 토큰 → `max_length=128` 로 충분합니다.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = 'monologg/koelectra-small-v3-discriminator'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def encode(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=128,
    )


train_ds = (
    Dataset.from_dict({'text': train_texts, 'label': train_labels})
    .map(encode, batched=True)
    .remove_columns(['text'])
)
val_ds = (
    Dataset.from_dict({'text': val_texts, 'label': val_labels})
    .map(encode, batched=True)
    .remove_columns(['text'])
)

print('학습 데이터셋:', train_ds)
print('검증 데이터셋:', val_ds)

## 6. 모델 + 평가 지표 + 클래스 가중치

이진 분류이므로 **Precision / Recall / F1** (양성 클래스 기준) 을 주요 지표로 봅니다.  
노이즈 vs 할 일 비율이 불균형할 수 있으므로 `compute_class_weight('balanced')` 로 보정합니다.

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

id2label = {0: '노이즈', 1: '할 일'}
label2id = {'노이즈': 0, '할 일': 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

_w = compute_class_weight('balanced', classes=np.array([0, 1]), y=train_labels)
_class_weights = torch.tensor(_w, dtype=torch.float)
print(f'클래스 가중치: 노이즈={_w[0]:.3f}, 할 일={_w[1]:.3f}')


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss_fn = torch.nn.CrossEntropyLoss(weight=_class_weights.to(logits.device))
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':  accuracy_score(labels, preds),
        'f1':        f1_score(labels, preds, pos_label=1, zero_division=0),
        'precision': precision_score(labels, preds, pos_label=1, zero_division=0),
        'recall':    recall_score(labels, preds, pos_label=1, zero_division=0),
    }

## 7. 학습 실행

에폭 10회, 배치 16. small 모델 기준 T4 GPU 약 5~10분.  
`metric_for_best_model='f1'` — 할 일 클래스 F1이 가장 높은 체크포인트를 자동 보존합니다.

In [ ]:
from transformers import TrainingArguments, DataCollatorWithPadding

args = TrainingArguments(
    output_dir='./koelectra-binary-output',
    save_safetensors=False,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to='none',
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

## 8. 최종 평가

검증셋에서 이진 분류 성능 확인. **할 일(1)** 클래스 F1이 발표 핵심 지표입니다.

In [ ]:
from sklearn.metrics import classification_report

preds_out = trainer.predict(val_ds)
y_pred = np.argmax(preds_out.predictions, axis=-1)
y_true = preds_out.label_ids

print(classification_report(
    y_true, y_pred,
    target_names=['노이즈', '할 일'],
    digits=4,
    zero_division=0,
))

## 9. 모델 저장

`predict.py` 가 `checkpoints/koelectra-binary/` 를 자동 참조합니다.  
이진 분류에서는 `labels.json` 이 불필요하므로 저장하지 않습니다.

In [ ]:
OUTPUT_DIR = './koelectra-binary'
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print('저장 완료:', OUTPUT_DIR)
!ls -lh {OUTPUT_DIR}

## 10. 추론 테스트

저장된 모델로 이진 추론을 검증합니다.  
**OCR 아티팩트 줄 필터** (`_OCR_LINE_NOISE`) 동작도 함께 확인합니다.

In [ ]:
# ── [Fix 1] OCR 아티팩트 필터 검증 ────────────────────────────────────────────
ocr_cases = [
    ('http://bit.ly/sarlang www.sarlang.com', True,  'URL 줄'),
    ('☎031-627-7916 (한컴지니케이)',          True,  '전화번호 줄'),
    ('19:00~20:00(예정)',                     True,  '시간 범위 줄'),
    ('→',                                    True,  '화살표 줄'),
    ('3월 20일까지 신청서를 제출해주세요.',   False, '정상 TODO 문장'),
]

print('[ OCR 아티팩트 줄 필터 (_OCR_LINE_NOISE) ]')
for line, expected, label in ocr_cases:
    got = bool(_OCR_LINE_NOISE.search(line))
    ok = got == expected
    print(f"  {'✅' if ok else '❌'} {label}: filtered={got}")

# ── 정규식 1차 필터 검증 ──────────────────────────────────────────────────────
def regex_filter_pass(sentence: str) -> bool:
    if len(sentence) < 7:
        return False
    return not any(re.search(p, sentence) for p in NON_TODO_PATTERNS)


filter_cases = [
    ('학부모님 안녕하세요.',                False, '인사말 제외'),
    ('안녕하세요.',                          False, '짧은 인사말'),
    ('2026. 4. 17.',                         False, '날짜 서명'),
    ('6월 5일까지 가정통신문 회신해주세요.', True,  '제출 TODO'),
    ('준비물: 실내화, 출입증 지참',          True,  '준비물 TODO'),
    ('5월 1일은 학교자율휴업일입니다.',      True,  '휴업일 공지'),
]

print('\n[ 정규식 1차 필터 검증 (NON_TODO_PATTERNS) ]')
for sent, expected, label in filter_cases:
    got = regex_filter_pass(sent)
    ok = got == expected
    print(f"  {'✅' if ok else '❌'} {label}: pass={got}")

# ── KoELECTRA 이진 추론 검증 ──────────────────────────────────────────────────
from transformers import AutoModelForSequenceClassification as AM

BINARY_THRESHOLD = 0.5
_tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
_clf = AM.from_pretrained(OUTPUT_DIR, num_labels=2)
_clf.eval()


def binary_predict(sentence: str) -> tuple[int, float]:
    """(label, prob_할일) 반환"""
    inputs = _tok(sentence, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        prob = torch.softmax(_clf(**inputs).logits, dim=-1)[0]
    label = 1 if prob[1].item() >= BINARY_THRESHOLD else 0
    return label, round(float(prob[1].item()), 3)


test_sents = [
    '학부모님 안녕하세요.',
    '4월 30일까지 체험학습 동의서를 제출해주세요.',
    '준비물은 도시락과 물병입니다.',
    '5월 1일은 학교자율휴업일입니다.',
    '서울갈산초등학교장',
]

print('\n[ KoELECTRA 이진 추론 결과 ]')
print(f'{"문장":<42} {"라벨":>6} {"P(할 일)":>10}')
print('-' * 62)
for s in test_sents:
    lbl, prob = binary_predict(s)
    tag = '할 일' if lbl == 1 else '노이즈'
    print(f'{s:<42} {tag:>6} {prob:>10.3f}')

## 11. 압축 + 다운로드

`koelectra-binary.zip` 을 `model/extraction/checkpoints/` 에서 풀면 `predict.py` 가 자동 로드합니다.

In [ ]:
!zip -r koelectra-binary.zip koelectra-binary/

from google.colab import files
files.download('koelectra-binary.zip')

## 끝

 → 압축 풀기 → 

### 수정 내역

| # | 문제 | 수정 내용 |
|---|------|----------|
| Fix 1 | split_sentences() OCR 필터 누락 | _OCR_LINE_NOISE 추가 — predict.py 와 동기화 |
| Fix 2 | is_todo 문서 단위 적용 오류 | JSONL 포맷 자동 감지 — 문장/문서 단위 분기 처리 |
| Fix 3 | 학습 데이터 v2 → v3 교체 | v3_dual_labeled_clean.jsonl (27,800행, is_todo+is_title 이중 라벨) |
